https://www.kaggle.com/competitions/store-sales-time-series-forecasting/overview/description
  
  Kolumna sales w train - sprzedaż konkretnej rodziny produktów, w konkretnym sklepie, konkretnego dnia. Twoim celem jest przewidzieć tę wartość dla 15 dni po końcu danych treningowych (to widać z opisu test.csv).
  
Święto oznaczone jako transferred = True formalnie wypada w tym dniu w kalendarzu, ale było świętowane innego dnia. Czyli jeśli po prostu weźmiesz holidays_events i powiesz "w te dni było święto", policzysz część dni błędnie - dzień przeniesionego święta zachowuje się jak zwykły dzień roboczy, a prawdziwe świętowanie było gdzie indziej, oznaczone osobnym wierszem z type = Transfer.
  
  Wypłaty w sektorze publicznym 15. i ostatniego dnia miesiąca - to sugeruje, że sprzedaż może mieć wzorzec dwutygodniowy powiązany z dniami wypłat, nie tylko standardową sezonowość tygodniową/miesięczną. To jest hipoteza, którą możesz przetestować na wykresie.
  
  Trzęsienie ziemi 16 kwietnia 2016 - to jest anomalia, jednorazowe zdarzenie, które prawdopodobnie mocno zaburzyło sprzedaż przez kilka tygodni (ludzie kupowali wodę i artykuły pierwszej potrzeby). Jeśli tego nie uwzględnisz, model może zinterpretować ten skok jako "normalny" wzorzec, co go zepsuje.

### Worked 2 years analyzing demand patterns in logistics - this applies the same framework to retail

In [81]:
import pandas as pd
import numpy as np
import os
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns

In [82]:
# Reading data
with open(r"C:\Users\krzeszutek\.kaggle\kaggle_token.txt", "r") as f:
    token  = f.read().strip()
os.environ['KAGGLE_API_TOKEN'] = token

# Download latest version
data_folder = kagglehub.competition_download('store-sales-time-series-forecasting') # ta funkcja wysłała żądanie do serwerów Kaggle, serwer odczytał ten klucz, zweryfikował moje konto i zezwolił na transfer danych.

print("Path to competition files:", data_folder)

ValueError: time data 'Mon, 22 Nov 2021 20:13:55 GMT' does not match format '%a, %d %b %Y %H:%M:%S %Z'

In [ ]:
# Listing all files names 
path_list = os.listdir(data_folder)
path_list

['holidays_events.csv',
 'oil.csv',
 'sample_submission.csv',
 'stores.csv',
 'test.csv',
 'train.csv',
 'transactions.csv']

In [ ]:
# Creating a dictionary to keep all tables
all_tables = {}

# Saving all files to dictionary by running a loop
for p in path_list:
    clear_table_name = p.replace(".csv", "")
    all_tables[clear_table_name] = pd.read_csv(os.path.join(data_folder, p))

all_tables.keys()

dict_keys(['holidays_events', 'oil', 'sample_submission', 'stores', 'test', 'train', 'transactions'])

#### Stores

In [ ]:
all_tables['stores'].columns

Index(['store_nbr', 'city', 'state', 'type', 'cluster'], dtype='object')

In [ ]:
all_tables['stores'].info()

# describe() - dla zmiennych numerycznych ciągłych
# value_counts() - dla zmiennych kategorycznych

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


In [ ]:
all_tables['stores']['city'].value_counts()

city
Quito            18
Guayaquil         8
Cuenca            3
Santo Domingo     3
Manta             2
Latacunga         2
Machala           2
Ambato            2
Quevedo           1
Esmeraldas        1
Loja              1
Libertad          1
Playas            1
Daule             1
Babahoyo          1
Salinas           1
Puyo              1
Guaranda          1
Ibarra            1
Riobamba          1
Cayambe           1
El Carmen         1
Name: count, dtype: int64

In [ ]:
all_tables['stores']['state'].value_counts()

state
Pichincha                         19
Guayas                            11
Santo Domingo de los Tsachilas     3
Azuay                              3
Manabi                             3
Cotopaxi                           2
Tungurahua                         2
Los Rios                           2
El Oro                             2
Chimborazo                         1
Imbabura                           1
Bolivar                            1
Pastaza                            1
Santa Elena                        1
Loja                               1
Esmeraldas                         1
Name: count, dtype: int64

In [ ]:
all_tables['stores']['type'].value_counts()

type
D    18
C    15
A     9
B     8
E     4
Name: count, dtype: int64

In [ ]:
all_tables['stores']['cluster'].value_counts()

cluster
3     7
6     6
10    6
15    5
13    4
14    4
11    3
4     3
8     3
1     3
9     2
7     2
2     2
12    1
5     1
16    1
17    1
Name: count, dtype: int64

In [ ]:
all_tables['stores'].isna().sum()

store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

#### Oil

In [ ]:
all_tables['oil'].columns

Index(['date', 'dcoilwtico'], dtype='object')

In [ ]:
all_tables['oil'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB


In [ ]:
all_tables['oil'].head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [ ]:
all_tables['oil']['date'].min()

'2013-01-01'

In [ ]:
all_tables['oil']['date'].max()

'2017-08-31'

In [ ]:
all_tables['oil']['date'].describe()

count           1218
unique          1218
top       2013-01-01
freq               1
Name: date, dtype: object

In [ ]:
all_tables['oil'].isna().sum()

date           0
dcoilwtico    43
dtype: int64

#### Sample_submission

In [ ]:
all_tables['sample_submission'].columns

Index(['id', 'sales'], dtype='object')

In [ ]:
all_tables['sample_submission'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      28512 non-null  int64  
 1   sales   28512 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 445.6 KB


In [ ]:
all_tables['sample_submission'].head()

,id,sales
0,3000888,0.0
1,3000889,0.0
2,3000890,0.0
3,3000891,0.0
4,3000892,0.0


In [ ]:
all_tables['sample_submission']['sales'].describe()

count    28512.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: sales, dtype: float64

In [ ]:
all_tables['sample_submission'].isna().sum()

id       0
sales    0
dtype: int64

#### Transactions

In [ ]:
all_tables['transactions'].columns

Index(['date', 'store_nbr', 'transactions'], dtype='object')

In [ ]:
all_tables['transactions'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB


In [ ]:
all_tables['transactions'].head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [ ]:
all_tables['transactions']['date'].min()

'2013-01-01'

In [ ]:
all_tables['transactions']['date'].max()

'2017-08-15'

In [ ]:
all_tables['transactions']['transactions'].describe()

count    83488.000000
mean      1694.602158
std        963.286644
min          5.000000
25%       1046.000000
50%       1393.000000
75%       2079.000000
max       8359.000000
Name: transactions, dtype: float64

In [ ]:
all_tables['transactions'].isna().sum()

date            0
store_nbr       0
transactions    0
dtype: int64

#### Holiday_events

In [ ]:
all_tables['holidays_events'].columns

Index(['date', 'type', 'locale', 'locale_name', 'description', 'transferred'], dtype='object')

In [ ]:
all_tables['holidays_events'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB


In [ ]:
all_tables['holidays_events'].head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [ ]:
all_tables['holidays_events']['date'].min()

'2012-03-02'

In [ ]:
all_tables['holidays_events']['date'].max()

'2017-12-26'

In [ ]:
all_tables['holidays_events']['type'].value_counts()

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

In [ ]:
all_tables['holidays_events']['locale'].value_counts()

locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

In [ ]:
all_tables['holidays_events']['locale_name'].value_counts()

locale_name
Ecuador                           174
Quito                              13
Riobamba                           12
Guaranda                           12
Latacunga                          12
Ambato                             12
Guayaquil                          11
Cuenca                              7
Ibarra                              7
Salinas                             6
Loja                                6
Santa Elena                         6
Santo Domingo de los Tsachilas      6
Quevedo                             6
Manta                               6
Esmeraldas                          6
Cotopaxi                            6
El Carmen                           6
Santo Domingo                       6
Machala                             6
Imbabura                            6
Puyo                                6
Libertad                            6
Cayambe                             6
Name: count, dtype: int64

In [ ]:
all_tables['holidays_events']['description'].value_counts()

description
Carnaval                              10
Fundacion de Cuenca                    7
Fundacion de Ibarra                    7
Fundacion de Quito                     6
Provincializacion de Santo Domingo     6
                                      ..
Terremoto Manabi+8                     1
Recupero puente Navidad                1
Terremoto Manabi+10                    1
Terremoto Manabi+11                    1
Traslado Fundacion de Quito            1
Name: count, Length: 103, dtype: int64

In [ ]:
all_tables['holidays_events']['transferred'].value_counts()

transferred
False    338
True      12
Name: count, dtype: int64

In [ ]:
all_tables['holidays_events'].isna().sum()

date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

#### Test

In [ ]:
all_tables['test'].columns

Index(['id', 'date', 'store_nbr', 'family', 'onpromotion'], dtype='object')

In [ ]:
all_tables['test'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           28512 non-null  int64 
 1   date         28512 non-null  object
 2   store_nbr    28512 non-null  int64 
 3   family       28512 non-null  object
 4   onpromotion  28512 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 1.1+ MB


In [ ]:
all_tables['test'].head()

,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


In [ ]:
all_tables['test']['date'].min()

'2017-08-16'

In [ ]:
all_tables['test']['date'].max()

'2017-08-31'

In [ ]:
all_tables['test']['family'].value_counts()

family
AUTOMOTIVE                    864
HOME APPLIANCES               864
SCHOOL AND OFFICE SUPPLIES    864
PRODUCE                       864
PREPARED FOODS                864
POULTRY                       864
PLAYERS AND ELECTRONICS       864
PET SUPPLIES                  864
PERSONAL CARE                 864
MEATS                         864
MAGAZINES                     864
LIQUOR,WINE,BEER              864
LINGERIE                      864
LAWN AND GARDEN               864
LADIESWEAR                    864
HOME CARE                     864
HOME AND KITCHEN II           864
BABY CARE                     864
HOME AND KITCHEN I            864
HARDWARE                      864
GROCERY II                    864
GROCERY I                     864
FROZEN FOODS                  864
EGGS                          864
DELI                          864
DAIRY                         864
CLEANING                      864
CELEBRATION                   864
BREAD/BAKERY                  864
BOOKS  

In [ ]:
all_tables['test']['onpromotion'].value_counts()

onpromotion
0      15943
1       2542
2       1102
3        648
10       557
       ...  
135        1
148        1
171        1
206        1
592        1
Name: count, Length: 212, dtype: int64

In [ ]:
all_tables['test'].isna().sum()

id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64

#### Train

In [ ]:
all_tables['train'].columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion'], dtype='object')

In [ ]:
all_tables['train'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


In [ ]:
all_tables['train'].head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [ ]:
all_tables['train']['date'].min()

'2013-01-01'

In [ ]:
all_tables['train']['date'].max()

'2017-08-15'

In [ ]:
all_tables['train']['family'].value_counts()

family
AUTOMOTIVE                    90936
HOME APPLIANCES               90936
SCHOOL AND OFFICE SUPPLIES    90936
PRODUCE                       90936
PREPARED FOODS                90936
POULTRY                       90936
PLAYERS AND ELECTRONICS       90936
PET SUPPLIES                  90936
PERSONAL CARE                 90936
MEATS                         90936
MAGAZINES                     90936
LIQUOR,WINE,BEER              90936
LINGERIE                      90936
LAWN AND GARDEN               90936
LADIESWEAR                    90936
HOME CARE                     90936
HOME AND KITCHEN II           90936
BABY CARE                     90936
HOME AND KITCHEN I            90936
HARDWARE                      90936
GROCERY II                    90936
GROCERY I                     90936
FROZEN FOODS                  90936
EGGS                          90936
DELI                          90936
DAIRY                         90936
CLEANING                      90936
CELEBRATION          

In [ ]:
all_tables['train']['onpromotion'].value_counts()

onpromotion
0      2389559
1       174551
2        79386
3        45862
4        31659
        ...   
313          1
452          1
642          1
305          1
425          1
Name: count, Length: 362, dtype: int64

In [ ]:
all_tables['train']['sales'].describe()

count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

In [ ]:
all_tables['train'].isna().sum()


id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64